# Quickstart: FD discovery with NeSyDep

Discover functional dependencies in a table with one call, compare algorithms, and export results.

In [1]:
import pandas as pd
import nesydep as nd

print('nesydep', nd.__version__)
print('algorithms:', nd.list_algorithms())

nesydep 0.1.3
algorithms: ['bsfd', 'ctane', 'dfd', 'pyref-fd', 'scfdm', 'tane']


## 1. A tiny hand-checkable table

zip determines city and state; city determines state.

In [2]:
df = pd.read_csv('../tests/data/toy_addresses.csv')
df.head()

,id,zip,city,state,gender,age_group
0,1,10001,New York,NY,F,20-29
1,2,10001,New York,NY,M,30-39
2,3,90001,Los Angeles,CA,F,20-29
3,4,90001,Los Angeles,CA,M,40-49
4,5,60601,Chicago,IL,F,30-39


In [3]:
# Exact mining with the pure-Python reference miner (no native build needed)
result = nd.discover(df, algo='pyref-fd')
for fd in result.fds:
    if 'id' not in fd.lhs and fd.rhs != 'id':  # hide key-column FDs
        print(fd)

[city] -> state
[city] -> zip
[state] -> city
[state] -> zip
[zip] -> city
[zip] -> state


[1/4] sampling (12 rows) ...
[2/4] correlation extraction ...
[3/4] partitioning ...
[4/4] mining (1 sub-tables) ...


## 2. Real data: BSFD vs TANE

On a wider table, exact full-table mining gets expensive — BSFD reduces the
search space first (stratified sampling + Bayesian-network correlation +
vertical partitioning), then mines sub-tables with the native kernel.

In [4]:
german = pd.read_csv('../tests/data/german_credit_sample.csv')
print(german.shape)

bsfd = nd.discover(german, algo='bsfd', support=5, confidence=0.95)
print(f'BSFD: {len(bsfd)} FDs, {bsfd.stats["n_subtables"]} sub-tables, '
      f'{bsfd.stats["mine_seconds"]:.2f}s mining')
for fd in bsfd.fds[:10]:
    print(' ', fd)

(300, 21)


[1/4] sampling (300 rows) ...
[2/4] correlation extraction ...


C:\Users\ruoch\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BSFD: 6 FDs, 17 sub-tables, 0.11s mining
  [Attribute10, Attribute11, Attribute12, Attribute17] -> Attribute20
  [Attribute10, Attribute11, Attribute12, Attribute9] -> Attribute20
  [Attribute10, Attribute11, Attribute7, Attribute9] -> Attribute20
  [Attribute10, Attribute11, Attribute12, Attribute19, Attribute7] -> Attribute20
  [Attribute11, Attribute12, Attribute14, Attribute17, Attribute18, Attribute19, Attribute4, Attribute7, Attribute9] -> Attribute15
  [Attribute10, Attribute12, Attribute6, Attribute7] -> Attribute20


[3/4] partitioning ...
[4/4] mining (17 sub-tables) ...


In [5]:
# Baseline TANE on a tractable column prefix (exact full-table mining is
# exponential in width — the very problem BSFD addresses)
tane = nd.discover(german.iloc[:, :16], algo='tane', support=5, confidence=1.0)
print(f'TANE: {len(tane)} FDs')

# BSFD's FDs live in the mined sub-tables; verify them against the full sample
from nesydep.evaluation.verify import verify
from nesydep.core.dataset import as_categorical_view, as_dataset

frame = as_categorical_view(as_dataset(german))
checks = verify(frame, bsfd.fds, min_support=5, min_confidence=0.9)
print(f'{sum(v.holds for v in checks)}/{len(checks)} BSFD FDs hold on the full sample')

[1/4] sampling (300 rows) ...
[2/4] correlation extraction ...
[3/4] partitioning ...
[4/4] mining (1 sub-tables) ...


TANE: 113 FDs
1/6 BSFD FDs hold on the full sample


In [6]:
# Export
bsfd.to_json('bsfd_fds.json')
bsfd.to_legacy_txt('bsfd_fds.txt')   # [A, B] -> C, one per line